# Revisiting Basics of Spacecraft Kinematics

Spacecraft attitude describes how the orientation of a rigid body changes relative to a chosen reference frame.

Unlike position, attitude cannot be represented by a single universally convenient coordinate set. Different representations such as direction cosine matrices, Euler angles, principal rotation vectors, Euler parameters, and Modified Rodrigues Parameters describe the same physical orientation in different mathematical forms, each with its own advantages and limitations.

Spacecraft kinematics is concerned with these attitude representations and with how they evolve in time. The central distinction is that **kinematics describes rotational motion without considering the torques that cause it**. The relationship between attitude and angular velocity is therefore established before the spacecraft equations of motion are introduced.

This module is primarily a rapid review of the spacecraft kinematics developed in the first *Spacecraft Dynamics and Control* specialization. It revisits:

- reference frames and attitude descriptions,
- direction cosine matrices and attitude composition,
- principal rotation representations,
- Euler parameters and Modified Rodrigues Parameters, and
- differential kinematic equations relating attitude coordinates to angular velocity.

The module therefore follows the progression

$$
\boxed{
\text{reference frames}
\;\rightarrow\;
\text{attitude representations}
\;\rightarrow\;
\text{attitude composition}
\;\rightarrow\;
\text{differential kinematics}
}
$$

These concepts provide the mathematical language required for the attitude dynamics and control developed later in the course.

This notebook focuses only on the concepts and subtleties required for the present course. Full derivations and the more detailed development of spacecraft attitude kinematics are available in the notes from the first specialization.

> **Prior material:**  
> [Specialization 1, Course 1 - Kinematics](https://github.com/johnm3398/Spacecraft-Dynamics-and-Control/tree/main/01_spacecraft_dynamics_and_control_specialization/01_kinematics)
>

---

In [1]:
import numpy as np

import matplotlib.pyplot as plt

from pathlib import Path

import sys
sys.path.insert(0, r"../../")
import AttitudeKinematicsLib as ak

In [2]:
print("Contents of AttitudeKinematicsLib:")
for name in sorted(dir(ak)):
    if not name.startswith("_"):
        print(name)

Contents of AttitudeKinematicsLib:
BInvmat_CRP
BInvmat_EP
BInvmat_Euler
BInvmat_MRP
BInvmat_PRV
Bmat_CRP
Bmat_EP
Bmat_Euler
Bmat_MRP
Bmat_PRV
CRP
CRP_to_DCM
DCM_to_CRP
DCM_to_EP
DCM_to_Euler
DCM_to_MRP
DCM_to_PRV
DCM_utils
EP_to_DCM
EulerAngles
EulerRodriguesParameters
Euler_to_DCM
MRP
MRP_compose
MRP_shadow
MRP_to_DCM
PRV
PRV_to_DCM
integrate_quaternion
normalize_quat
np
quat_derivative
quat_diff
quat_inv
quat_mult
rotation_matrix_x
rotation_matrix_y
rotation_matrix_z
skew_symmetric
solve_ivp
validate_DCM
validate_vec3
validate_vec4


# 1 - Particle Kinematics Fundamentals

# 2 - Fundamentals of Attitude Kinetics

## 2.1 - DCM

A **Direction Cosine Matrix (DCM)** is probably the most fundamental attitude representation to keep in your head.

Quaternions, MRPs, PRVs, Euler angles, etc. are all different ways of packaging an attitude. A DCM is the common language they can all be converted to and from.

It is basically the **LEGO baseplate of attitude representations**: you can build the attitude using different pieces, but sooner or later they can all snap onto the same $3\times3$ representation.

> A DCM is not necessarily the most compact or convenient attitude representation, but it is one of the cleanest ways to understand what an attitude transformation actually does.


**<ins>Frames and Basis Vectors</ins>**

A coordinate frame $\mathcal B$ is defined by an orthonormal basis

$$
\mathcal B = \left\{\hat{\mathbf b}_1,\hat{\mathbf b}_2,\hat{\mathbf b}_3\right\}.
$$

These basis vectors exist geometrically on their own, but the moment we write their numerical components, we need to say **with respect to which frame are we describing them**.

For example,

$$
{}^N\hat{\mathbf b}_1
$$

means:

> the first basis vector of frame $\mathcal B$, expressed using coordinates from frame $\mathcal N$.

Suppose all three body axes are written in inertial-frame components:

$$
{}^N\hat{\mathbf b}_1,\qquad {}^N\hat{\mathbf b}_2,\qquad {}^N\hat{\mathbf b}_3.
$$

Stacking them as columns gives

$$
\boxed{
C_{NB} =
\begin{bmatrix}
{}^N\hat{\mathbf b}_1 &
{}^N\hat{\mathbf b}_2 &
{}^N\hat{\mathbf b}_3
\end{bmatrix}
}
$$

where $C_{NB}$ maps vector components from $\mathcal B$ into $\mathcal N$:

$$
\boxed{
{}^N\mathbf v = C_{NB}\,{}^B\mathbf v.
}
$$

The columns therefore answer a very physical question:

> **Where do the axes of frame $\mathcal B$ point when viewed from frame $\mathcal N$?**


**<ins>Why the Basis Vectors Appear as Columns</ins>**

Take a vector written in body-frame components,

$$
{}^B\mathbf v =
\begin{bmatrix}
v_1\\
v_2\\
v_3
\end{bmatrix}.
$$

Geometrically,

$$
\mathbf v = v_1\hat{\mathbf b}_1 + v_2\hat{\mathbf b}_2 + v_3\hat{\mathbf b}_3.
$$

Writing that same vector using inertial-frame components gives

$$
{}^N\mathbf v = v_1\,{}^N\hat{\mathbf b}_1 + v_2\,{}^N\hat{\mathbf b}_2 + v_3\,{}^N\hat{\mathbf b}_3.
$$

That is exactly

$$
{}^N\mathbf v =
\begin{bmatrix}
{}^N\hat{\mathbf b}_1 &
{}^N\hat{\mathbf b}_2 &
{}^N\hat{\mathbf b}_3
\end{bmatrix}
{}^B\mathbf v.
$$

So the column structure is not some random convention. It falls straight out of how a vector is reconstructed from its basis vectors.


**<ins>The Transpose Gives the Reverse Transformation</ins>**

A valid DCM is orthogonal:

$$
C_{NB}^T C_{NB} = I.
$$

Therefore,

$$
C_{NB}^{-1} = C_{NB}^T.
$$

So if

$$
{}^N\mathbf v = C_{NB}\,{}^B\mathbf v,
$$

then

$$
{}^B\mathbf v = C_{NB}^T\,{}^N\mathbf v.
$$

Define

$$
\boxed{
C_{BN} = C_{NB}^T.
}
$$

Hence,

$$
\boxed{
{}^B\mathbf v = C_{BN}\,{}^N\mathbf v.
}
$$

This also gives a nice way of reading the rows of the matrix:

$$
C_{BN} =
\begin{bmatrix}
({}^N\hat{\mathbf b}_1)^T\\
({}^N\hat{\mathbf b}_2)^T\\
({}^N\hat{\mathbf b}_3)^T
\end{bmatrix}.
$$

Each row effectively asks:

> **How much of this vector lies along one of my body axes?**

So a useful mental model is

$$
\boxed{
\text{columns} \rightarrow \text{where the source-frame axes point}
}
$$

$$
\boxed{
\text{rows} \rightarrow \text{how to extract components along those axes}
}
$$

**<ins>DCM Properties</ins>**

A proper DCM belongs to the rotation group $SO(3)$ and satisfies

$$
\boxed{
C^T C = CC^T = I
}
$$

and

$$
\boxed{
\det(C) = +1.
}
$$

Its rows and columns are therefore orthonormal:

$$
\|\mathbf c_i\| = 1,
$$

$$
\mathbf c_i^T\mathbf c_j = 0,\qquad i\neq j.
$$

A DCM also preserves lengths and angles. If

$$
{}^A\mathbf v = C_{AB}\,{}^B\mathbf v,
$$

then

$$
\|{}^A\mathbf v\| = \|{}^B\mathbf v\|.
$$

The physical vector has not changed. We have only changed the coordinates used to describe it.

Although a DCM contains nine numbers, attitude has only three degrees of freedom. The orthogonality constraints remove the redundant information.

**<ins>Composing Attitudes</ins>**

Frame transformations chain naturally.

Suppose

$$
{}^N\mathbf v = C_{NB_1}\,{}^{B_1}\mathbf v
$$

and

$$
{}^{B_1}\mathbf v = C_{B_1B_2}\,{}^{B_2}\mathbf v.
$$

Then

$$
{}^N\mathbf v = C_{NB_1}C_{B_1B_2}\,{}^{B_2}\mathbf v,
$$

so

$$
\boxed{
C_{NB_2} = C_{NB_1}C_{B_1B_2}.
}
$$

The middle frame almost behaves like units cancelling:

$$
N\leftarrow B_1,\qquad B_1\leftarrow B_2
$$

gives

$$
N\leftarrow B_2.
$$

This is a very useful sanity check when chaining multiple frame transformations.

**<ins>Differential Kinematic Equation</ins>**

A DCM tells us the attitude of one frame relative to another. If the body rotates, the DCM must change with time. The differential kinematic equation links that change directly to the angular velocity.

Take a vector $\mathbf v$ that is fixed in the inertial frame $\mathcal N$:

$$
{}^N\dot{\mathbf v} = \mathbf 0.
$$

Its components in the body frame are

$$
{}^B\mathbf v = C_{BN}\,{}^N\mathbf v.
$$

Since the body frame is rotating with angular velocity $\boldsymbol{\omega}_{B/N}$, the same inertially fixed vector appears to move in the opposite direction when viewed from $\mathcal B$:

$$
{}^B\dot{\mathbf v} = -\boldsymbol{\omega}_{B/N}\times{}^B\mathbf v.
$$

Using the skew-symmetric matrix,

$$
\boldsymbol{\omega}\times\mathbf v = [\tilde{\boldsymbol{\omega}}]\mathbf v,
$$

gives

$$
{}^B\dot{\mathbf v} = -[\tilde{\boldsymbol{\omega}}_{B/N}]\,{}^B\mathbf v.
$$

From

$$
{}^B\mathbf v = C_{BN}\,{}^N\mathbf v,
$$

and because ${}^N\mathbf v$ is constant,

$$
{}^B\dot{\mathbf v} = \dot C_{BN}\,{}^N\mathbf v.
$$

Substituting ${}^B\mathbf v = C_{BN}\,{}^N\mathbf v$ into the rotational relation gives

$$
\dot C_{BN}\,{}^N\mathbf v = -[\tilde{\boldsymbol{\omega}}_{B/N}]\,C_{BN}\,{}^N\mathbf v.
$$

Since this must hold for any vector ${}^N\mathbf v$,

$$
\boxed{\dot C_{BN} = -[\tilde{\boldsymbol{\omega}}_{B/N}]\,C_{BN}}.
$$

The minus sign has a simple physical meaning: if the body rotates one way, an inertially fixed vector appears to rotate the opposite way when viewed from the body frame.

For the reverse transformation,

$$
C_{NB} = C_{BN}^T.
$$

Differentiating,

$$
\dot C_{NB} = \dot C_{BN}^T.
$$

Using the previous result,

$$
\dot C_{NB} = \left(-[\tilde{\boldsymbol{\omega}}_{B/N}]\,C_{BN}\right)^T.
$$

Using $(AB)^T = B^TA^T$,

$$
\dot C_{NB} = C_{BN}^T\left(-[\tilde{\boldsymbol{\omega}}_{B/N}]\right)^T.
$$

Since a skew-symmetric matrix satisfies

$$
[\tilde{\boldsymbol{\omega}}]^T = -[\tilde{\boldsymbol{\omega}}],
$$

we get

$$
\boxed{\dot C_{NB} = C_{NB}[\tilde{\boldsymbol{\omega}}_{B/N}]}.
$$

So the two equivalent DCM kinematic equations are

$$
\boxed{\dot C_{BN} = -[\tilde{\boldsymbol{\omega}}_{B/N}]\,C_{BN}},
$$

$$
\boxed{\dot C_{NB} = C_{NB}[\tilde{\boldsymbol{\omega}}_{B/N}]}.
$$

In exact mathematics, propagating the DCM this way preserves

$$
C^TC = I,\qquad \det(C)=1.
$$

In numerical simulations, small integration errors can slowly break these constraints, which is one reason quaternions or MRPs are often preferred for attitude propagation.

**<ins>DCMs as the Common Attitude Interface</ins>**

Different attitude representations are useful for different jobs:

- Euler angles are intuitive but can be singular.
- Quaternions are compact and convenient for propagation.
- MRPs are minimal and useful for nonlinear control.
- PRVs give a very geometric axis-angle description.

But they all describe the same physical orientation.

A DCM acts as a common interface between them:

$$
\text{Quaternion}
\leftrightarrow
\boxed{\text{DCM}}
\leftrightarrow
\text{MRP}
$$

$$
\text{PRV}
\leftrightarrow
\boxed{\text{DCM}}
\leftrightarrow
\text{Euler Angles}.
$$

So while the DCM is not always the representation we want to propagate or control with, it is a really useful **ground truth representation** for thinking about frame transformations.

If I am ever confused about what an attitude representation actually means, converting it back to a DCM usually brings the geometry back into focus.

## 2.2 - PRV

In [3]:
# Quiz 5 - Question 2
BN = np.array([[0.0, 1.0, 0.0], 
               [0.0, 0.0, 1.0], 
               [1.0, 0.0, 0.0]])

ak.DCM_to_PRV(BN)

(array([0.57735027, 0.57735027, 0.57735027]), 120.00000000000001)

## 2.3 - Euler Parameter (Quaternion)

In [4]:
# Quiz 6 - Question 3
BN = np.array([[0.0, 1.0, 0.0], 
               [0.0, 0.0, 1.0], 
               [1.0, 0.0, 0.0]])

ak.DCM_to_EP(BN)

array([0.5, 0.5, 0.5, 0.5])

## 2.4 - MRPs

In [5]:
# Quiz 7 - Question 2
sigma_BN_1 = (1/3) * np.array([1.0, 1.0, 1.0])
sigma_BN_2 = -(1/3) * np.array([1.0, 1.0, 1.0])

sigma_RN = (1/3) * np.array([-1.0, 1.0, -1.0])

sigma_BR_1 = ak.MRP_compose(sigma_BN_1, sigma_RN, "sub")
sigma_BR_2 = ak.MRP_compose(sigma_BN_2, sigma_RN, "sub")

print("sigma_BR_1:", sigma_BR_1)
print("sigma_BR_2:", sigma_BR_2)

sigma_BR_1: [6.24500451e-17 0.00000000e+00 1.00000000e+00]
sigma_BR_2: [ 0.33333333 -0.33333333 -0.33333333]
